In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

## Задача преобразовать данные из файла bike_share/weather.csv, для работы с гипотезами

- читаю файл bike_share/weather.csv

In [2]:
weather = pd.read_csv('../../bike_share/weather.csv')
weather

,name,datetime,tempmax,tempmin,temp,feelslikemax,feelslikemin,feelslike,dew,humidity,...,solarenergy,uvindex,severerisk,sunrise,sunset,moonphase,conditions,description,icon,stations
0,"Washington,DC,USA",2020-05-01,18.8,11.6,14.9,18.8,11.6,14.9,8.9,69.6,...,9.5,6,NaN,2020-05-01T06:09:41,2020-05-01T20:01:16,0.30,"Rain, Partially cloudy",Partly cloudy throughout the day with rain cle...,rain,"KDCA,72405013743,72403793728,F0198,KADW,KDAA,7..."
1,"Washington,DC,USA",2020-05-02,22.1,11.1,16.3,22.1,11.1,16.3,6.4,54.0,...,14.0,9,NaN,2020-05-02T06:08:30,2020-05-02T20:02:13,0.33,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,"KDCA,72405013743,72403793728,F0198,KGAI,KADW,K..."
2,"Washington,DC,USA",2020-05-03,24.9,15.6,18.6,24.9,15.6,18.6,13.4,72.5,...,5.7,4,NaN,2020-05-03T06:07:21,2020-05-03T20:03:11,0.37,"Rain, Partially cloudy",Partly cloudy throughout the day with rain.,rain,"KDCA,72405013743,72403793728,F0198,KADW,KDAA,7..."
3,"Washington,DC,USA",2020-05-04,23.8,14.3,19.2,23.8,14.3,19.2,7.8,53.8,...,13.3,8,NaN,2020-05-04T06:06:12,2020-05-04T20:04:08,0.40,"Rain, Partially cloudy",Clearing in the afternoon with early morning r...,rain,"KDCA,72405013743,72403793728,F0198,KADW,KDAA,7..."
4,"Washington,DC,USA",2020-05-05,14.3,9.3,12.3,14.3,8.0,12.1,3.3,55.6,...,5.0,2,NaN,2020-05-05T06:05:06,2020-05-05T20:05:05,0.44,"Rain, Partially cloudy",Partly cloudy throughout the day with late aft...,rain,"KIAD,KDCA,72405013743,72403793728,72403093738,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1579,"Washington,DC,USA",2024-08-27,33.3,22.5,27.7,33.7,22.5,27.9,18.9,62.4,...,18.1,8,10.0,2024-08-27T06:33:04,2024-08-27T19:45:18,0.78,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,"KDCA,72405013743,D6279,72403793728,KGAI,KADW,K..."
1580,"Washington,DC,USA",2024-08-28,37.8,24.2,30.2,39.7,24.2,31.9,20.4,58.7,...,16.9,7,10.0,2024-08-28T06:33:58,2024-08-28T19:43:49,0.81,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,"KDCA,72405013743,D6279,72403793728,KADW,KDAA,7..."
1581,"Washington,DC,USA",2024-08-29,32.8,23.3,28.1,38.1,23.3,30.3,21.9,69.7,...,11.5,8,60.0,2024-08-29T06:34:52,2024-08-29T19:42:19,0.85,"Rain, Partially cloudy",Partly cloudy throughout the day with late aft...,rain,"KDCA,72405013743,D6279,72403793728,KGAI,KADW,K..."
1582,"Washington,DC,USA",2024-08-30,23.8,22.2,22.9,23.8,22.2,22.9,20.8,87.6,...,3.4,2,10.0,2024-08-30T06:35:45,2024-08-30T19:40:48,0.88,"Rain, Overcast",Cloudy skies throughout the day with rain.,rain,"KDCA,72405013743,D6279,72403793728,KGAI,KADW,K..."


- смотрю типы данных

In [3]:
weather.dtypes

name                    str
datetime                str
tempmax             float64
tempmin             float64
temp                float64
feelslikemax        float64
feelslikemin        float64
feelslike           float64
dew                 float64
humidity            float64
precip              float64
precipprob            int64
precipcover         float64
preciptype              str
snow                float64
snowdepth           float64
windgust            float64
windspeed           float64
winddir             float64
sealevelpressure    float64
cloudcover          float64
visibility          float64
solarradiation      float64
solarenergy         float64
uvindex               int64
severerisk          float64
sunrise                 str
sunset                  str
moonphase           float64
conditions              str
description             str
icon                    str
stations                str
dtype: object

- меняю тип данных для столбца datetime

In [4]:
weather['datetime'] = pd.to_datetime(weather['datetime'], unit='ns')

- выбираю нужные столбцы, которые я выбрал еще в этапе исследование данных файла bike_share/weather.csv
    - temp
    - humidity
    - dew
    - visibility
    - windspeed
    - precipcover
    - snow

In [5]:
weather = weather[['datetime', 'temp', 'humidity', 'dew', 'visibility', 'windspeed', 'precipcover', 'snow', 'icon']]
weather

,datetime,temp,humidity,dew,visibility,windspeed,precipcover,snow,icon
0,2020-05-01,14.9,69.6,8.9,16.0,25.7,29.17,0.0,rain
1,2020-05-02,16.3,54.0,6.4,15.9,26.1,0.00,0.0,partly-cloudy-day
2,2020-05-03,18.6,72.5,13.4,15.2,21.4,16.67,0.0,rain
3,2020-05-04,19.2,53.8,7.8,15.2,44.9,12.50,0.0,rain
4,2020-05-05,12.3,55.6,3.3,15.2,24.0,16.67,0.0,rain
...,...,...,...,...,...,...,...,...,...
1579,2024-08-27,27.7,62.4,18.9,15.1,17.0,0.00,0.0,partly-cloudy-day
1580,2024-08-28,30.2,58.7,20.4,16.0,23.1,0.00,0.0,partly-cloudy-day
1581,2024-08-29,28.1,69.7,21.9,14.9,23.3,25.00,0.0,rain
1582,2024-08-30,22.9,87.6,20.8,12.8,19.8,20.83,0.0,rain


- теперь данные выше нужно соединить с данными из файла bike_share/daily_rent_detail.csv

- читаю файл bike_share/daily_rent_detail.csv

In [6]:
daily_rent_detail = pd.read_csv('../../bike_share/daily_rent_detail.csv')
daily_rent_detail

C:\Users\lenov\AppData\Local\Temp\ipykernel_18092\2254643125.py:1: DtypeWarning: Columns (0: start_station_id, 1: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  daily_rent_detail = pd.read_csv('../../bike_share/daily_rent_detail.csv')


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,946D42AD89539210,docked_bike,2020-05-30 17:25:29,2020-05-31 18:25:22,Anacostia Library,31804,11th & H St NE,31614.0,38.865784,-76.978400,38.899983,-76.991383,casual
1,CC46FAAB662B8613,docked_bike,2020-05-09 14:42:04,2020-05-09 15:06:33,10th & E St NW,31256,21st St & Constitution Ave NW,31261.0,38.895914,-77.026064,38.892459,-77.046567,member
2,72F00B2FB833D6ED,docked_bike,2020-05-24 17:27:19,2020-05-24 17:43:51,Connecticut Ave & Newark St NW / Cleveland Park,31305,12th & U St NW,31268.0,38.934267,-77.057979,38.916787,-77.028139,member
3,4DFBE6AED989DF35,docked_bike,2020-05-27 15:29:52,2020-05-27 15:47:13,Connecticut Ave & Newark St NW / Cleveland Park,31305,14th & Belmont St NW,31119.0,38.934267,-77.057979,38.921074,-77.031887,casual
4,1AAFE6B4331AB9DF,docked_bike,2020-05-31 14:06:03,2020-05-31 14:30:30,Georgia Ave & Morton St NW,31419,17th & K St NW,31213.0,38.932128,-77.023500,38.902760,-77.038630,casual
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16086667,117C27DBDD138B72,electric_bike,2024-08-01 08:10:22.293,2024-08-01 08:10:37.641,NaN,NaN,NaN,NaN,38.890000,-77.000000,38.890000,-77.000000,member
16086668,4774F4D630258482,electric_bike,2024-08-08 10:05:21.938,2024-08-08 10:21:46.654,NaN,NaN,NaN,NaN,38.900000,-77.000000,38.870000,-76.950000,member
16086669,D75836E25E77B5EC,electric_bike,2024-08-03 16:29:32.252,2024-08-03 16:35:43.179,NaN,NaN,NaN,NaN,38.920000,-76.990000,38.920000,-77.000000,member
16086670,3B888603D18116DC,electric_bike,2024-08-03 02:49:45.661,2024-08-03 02:59:56.877,NaN,NaN,NaN,NaN,38.920000,-77.020000,38.920000,-77.030000,member


- добавлю новый столбец date

In [7]:
daily_rent_detail['date'] = pd.to_datetime(daily_rent_detail['started_at'], format='mixed').dt.date

- меняю тип данных для столбца date 

In [8]:
daily_rent_detail['date'] = pd.to_datetime(daily_rent_detail['date'], unit='ns')

- меняю название столбца в dataframe weather

In [9]:
weather.rename(columns={'datetime': 'date'}, inplace=True)

- смотрю результат

In [10]:
weather.columns

Index(['date', 'temp', 'humidity', 'dew', 'visibility', 'windspeed',
       'precipcover', 'snow', 'icon'],
      dtype='str')

- сделал merge для таблицы daily_rent_detail соединив ее с таблицей weather

In [11]:
daily_rent_detail_with_weather = daily_rent_detail.merge(weather, how='inner', on='date')
daily_rent_detail_with_weather

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,member_casual,date,temp,humidity,dew,visibility,windspeed,precipcover,snow,icon
0,946D42AD89539210,docked_bike,2020-05-30 17:25:29,2020-05-31 18:25:22,Anacostia Library,31804,11th & H St NE,31614.0,38.865784,-76.978400,...,casual,2020-05-30,24.3,55.7,13.8,16.0,32.7,4.17,0.0,rain
1,CC46FAAB662B8613,docked_bike,2020-05-09 14:42:04,2020-05-09 15:06:33,10th & E St NW,31256,21st St & Constitution Ave NW,31261.0,38.895914,-77.026064,...,member,2020-05-09,6.8,41.9,-5.7,16.0,40.8,0.00,0.0,partly-cloudy-day
2,72F00B2FB833D6ED,docked_bike,2020-05-24 17:27:19,2020-05-24 17:43:51,Connecticut Ave & Newark St NW / Cleveland Park,31305,12th & U St NW,31268.0,38.934267,-77.057979,...,member,2020-05-24,18.0,82.3,14.9,15.8,20.9,0.00,0.0,cloudy
3,4DFBE6AED989DF35,docked_bike,2020-05-27 15:29:52,2020-05-27 15:47:13,Connecticut Ave & Newark St NW / Cleveland Park,31305,14th & Belmont St NW,31119.0,38.934267,-77.057979,...,casual,2020-05-27,22.2,79.2,18.2,15.8,17.5,0.00,0.0,partly-cloudy-day
4,1AAFE6B4331AB9DF,docked_bike,2020-05-31 14:06:03,2020-05-31 14:30:30,Georgia Ave & Morton St NW,31419,17th & K St NW,31213.0,38.932128,-77.023500,...,casual,2020-05-31,20.4,39.5,6.0,16.0,28.5,0.00,0.0,partly-cloudy-day
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16086667,117C27DBDD138B72,electric_bike,2024-08-01 08:10:22.293,2024-08-01 08:10:37.641,NaN,NaN,NaN,NaN,38.890000,-77.000000,...,member,2024-08-01,29.6,64.5,21.6,15.9,17.1,0.00,0.0,partly-cloudy-day
16086668,4774F4D630258482,electric_bike,2024-08-08 10:05:21.938,2024-08-08 10:21:46.654,NaN,NaN,NaN,NaN,38.900000,-77.000000,...,member,2024-08-08,25.9,86.0,23.4,13.6,20.0,50.00,0.0,rain
16086669,D75836E25E77B5EC,electric_bike,2024-08-03 16:29:32.252,2024-08-03 16:35:43.179,NaN,NaN,NaN,NaN,38.920000,-76.990000,...,member,2024-08-03,27.9,71.1,21.9,15.7,34.6,12.50,0.0,rain
16086670,3B888603D18116DC,electric_bike,2024-08-03 02:49:45.661,2024-08-03 02:59:56.877,NaN,NaN,NaN,NaN,38.920000,-77.020000,...,member,2024-08-03,27.9,71.1,21.9,15.7,34.6,12.50,0.0,rain


- вывел название столбцов

In [12]:
daily_rent_detail_with_weather.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual', 'date', 'temp', 'humidity', 'dew', 'visibility',
       'windspeed', 'precipcover', 'snow', 'icon'],
      dtype='str')

- оставляю нужные столбцы

In [13]:
drd_with_weather = daily_rent_detail_with_weather[
    [
        'rideable_type', 'started_at', 'ended_at', 'member_casual', 'date',
        'temp', 'humidity', 'dew', 'visibility', 'windspeed', 'precipcover', 'snow', 'icon'
       ]
]
drd_with_weather

,rideable_type,started_at,ended_at,member_casual,date,temp,humidity,dew,visibility,windspeed,precipcover,snow,icon
0,docked_bike,2020-05-30 17:25:29,2020-05-31 18:25:22,casual,2020-05-30,24.3,55.7,13.8,16.0,32.7,4.17,0.0,rain
1,docked_bike,2020-05-09 14:42:04,2020-05-09 15:06:33,member,2020-05-09,6.8,41.9,-5.7,16.0,40.8,0.00,0.0,partly-cloudy-day
2,docked_bike,2020-05-24 17:27:19,2020-05-24 17:43:51,member,2020-05-24,18.0,82.3,14.9,15.8,20.9,0.00,0.0,cloudy
3,docked_bike,2020-05-27 15:29:52,2020-05-27 15:47:13,casual,2020-05-27,22.2,79.2,18.2,15.8,17.5,0.00,0.0,partly-cloudy-day
4,docked_bike,2020-05-31 14:06:03,2020-05-31 14:30:30,casual,2020-05-31,20.4,39.5,6.0,16.0,28.5,0.00,0.0,partly-cloudy-day
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16086667,electric_bike,2024-08-01 08:10:22.293,2024-08-01 08:10:37.641,member,2024-08-01,29.6,64.5,21.6,15.9,17.1,0.00,0.0,partly-cloudy-day
16086668,electric_bike,2024-08-08 10:05:21.938,2024-08-08 10:21:46.654,member,2024-08-08,25.9,86.0,23.4,13.6,20.0,50.00,0.0,rain
16086669,electric_bike,2024-08-03 16:29:32.252,2024-08-03 16:35:43.179,member,2024-08-03,27.9,71.1,21.9,15.7,34.6,12.50,0.0,rain
16086670,electric_bike,2024-08-03 02:49:45.661,2024-08-03 02:59:56.877,member,2024-08-03,27.9,71.1,21.9,15.7,34.6,12.50,0.0,rain


- подготовлю типы данных всех столбцов

In [14]:
drd_with_weather = drd_with_weather.astype(
    {
        'rideable_type': 'object',
        'started_at': 'datetime64[ns]',
        'ended_at': 'datetime64[ns]',
        'member_casual': 'object',
        'date': 'datetime64[us]',
        'temp': 'float32',
        'humidity': 'float32',
        'dew': 'float32',
        'visibility': 'float32',
        'windspeed': 'float32',
        'precipcover': 'float32',
        'snow': 'float32',
        'icon': 'object'
    }
)

- добавляю столбец duration_in_minutes, он будет нужен для определения продолжительности поездки
- сразу сделаю в минутах

In [15]:
drd_with_weather['duration_in_minutes'] = drd_with_weather['ended_at'] - drd_with_weather['started_at']
drd_with_weather['duration_in_minutes'] = drd_with_weather['duration_in_minutes'].dt.total_seconds().div(60)

- убираю отрицательные значения в столбце duration_in_minutes

In [17]:
drd_with_weather = drd_with_weather[drd_with_weather['duration_in_minutes'] >= 0]

- меняю тип данных для столбца duration_in_minutes

In [21]:
drd_with_weather = drd_with_weather.astype({'duration_in_minutes': 'float32'})

- записываю dataframe drd_with_weather в файл parquet

In [22]:
table_for_parquet = pa.Table.from_pandas(drd_with_weather)

In [23]:
pq.write_table(table_for_parquet, '../../data_after_preprocessing/drd_with_weather.parquet')